In [ ]:
import joblib
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Hàm để lưu mô hình học máy, scaler và OneHotEncoder
def save_model_scaler_and_encoder(model, scaler, encoder, model_filename):
    """
    Lưu mô hình học máy, scaler và OneHotEncoder vào một file duy nhất.
    
    Parameters:
    - model: Mô hình học máy đã huấn luyện (ví dụ: RandomForestClassifier, LogisticRegression,...)
    - scaler: Scaler đã được huấn luyện (ví dụ: StandardScaler)
    - encoder: OneHotEncoder đã được huấn luyện
    - model_filename: Tên file để lưu mô hình, scaler và encoder.
    """
    # Lưu vào một file duy nhất (sử dụng joblib để lưu mô hình, scaler và encoder)
    joblib.dump((model, scaler, encoder), model_filename)
    print(f'Model, scaler, and encoder have been saved to {model_filename}')

# Ví dụ sử dụng
if __name__ == "__main__":
    # Load dữ liệu Iris
    data = load_iris()
    X = data.data
    y = data.target

    # Chia dữ liệu thành train và test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    # OneHotEncoding cho các cột phân loại (ví dụ: giả sử cột 0 là phân loại)
    encoder = OneHotEncoder(sparse=False)
    # Chỉ cần one-hot cho các cột phân loại trong X (ví dụ, giả sử cột đầu tiên là phân loại)
    X_train_encoded = encoder.fit_transform(X_train[:, [0]])
    X_test_encoded = encoder.transform(X_test[:, [0]])
    
    # Dữ liệu X sau khi one-hot encoding
    X_train_encoded_full = np.hstack((X_train_encoded, X_train[:, 1:]))
    X_test_encoded_full = np.hstack((X_test_encoded, X_test[:, 1:]))

    # Tiền xử lý - Sử dụng StandardScaler để chuẩn hóa dữ liệu
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_encoded_full)
    X_test_scaled = scaler.transform(X_test_encoded_full)

    # Huấn luyện mô hình RandomForest
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)

    # Lưu mô hình, scaler và encoder vào một file duy nhất
    save_model_scaler_and_encoder(model, scaler, encoder, 'model_scaler_encoder.pkl')
Giải thích:
One-Hot Encoding:

Chúng ta sử dụng OneHotEncoder từ sklearn.preprocessing để chuyển đổi một hoặc nhiều cột phân loại thành dạng One-Hot. Trong ví dụ này, giả sử cột đầu tiên là phân loại.

sparse=False giúp OneHotEncoder trả về mảng NumPy thay vì ma trận sparse, giúp dễ dàng thao tác hơn trong một số trường hợp.

Scaler (StandardScaler):

StandardScaler được sử dụng để chuẩn hóa dữ liệu sau khi đã thực hiện One-Hot Encoding.

Mô hình học máy:

Sử dụng RandomForestClassifier làm mô hình học máy. Bạn có thể thay thế mô hình này bằng bất kỳ mô hình nào khác tuỳ theo bài toán.

Lưu mô hình, scaler và encoder:

Hàm save_model_scaler_and_encoder sẽ lưu tất cả các thành phần đã huấn luyện (mô hình, scaler và encoder) vào một file duy nhất.

Thư viện joblib được sử dụng để lưu trữ chúng dưới dạng một file pkl.

Cách tải lại mô hình, scaler và encoder:
Để tải lại mô hình, scaler và encoder từ file đã lưu, bạn có thể sử dụng hàm joblib.load() như sau:

python
Copy code
def load_model_scaler_and_encoder(filename):
    # Tải lại mô hình, scaler và encoder từ file
    model, scaler, encoder = joblib.load(filename)
    print(f'Model, scaler, and encoder have been loaded from {filename}')
    return model, scaler, encoder

# Ví dụ sử dụng:
if __name__ == "__main__":
    # Tải lại mô hình, scaler và encoder
    model, scaler, encoder = load_model_scaler_and_encoder('model_scaler_encoder.pkl')
    
    # Dùng mô hình và scaler đã tải lại để dự đoán
    X_new = [[5.1, 3.5, 1.4, 0.2]]  # Dữ liệu mẫu mới
    # Chuyển đổi dữ liệu mới qua One-Hot Encoding
    X_new_encoded = encoder.transform([[5.1]])  # Chỉ mã hóa cột phân loại đầu tiên
    X_new_full = np.hstack((X_new_encoded, X_new[:, 1:]))  # Thêm các cột còn lại
    X_new_scaled = scaler.transform(X_new_full)  # Chuẩn hóa dữ liệu mới
    prediction = model.predict(X_new_scaled)  # Dự đoán
    print(f'Prediction: {prediction}')

In [ ]:
from category_encoders import TargetEncoder

def objective_logreg(trial):

    C = trial.suggest_float("C", 1e-3, 10.0, log=True)

    oof_pred = np.zeros(len(X_sel))
    y_array = np.array(y)

    for tr_idx, va_idx in folds.split(X_sel, y):
        X_tr, X_va = X_sel.iloc[tr_idx], X_sel.iloc[va_idx]
        y_tr, y_va = y_array[tr_idx], y_array[va_idx]

        # Target Encoding
        te = TargetEncoder()
        X_tr_enc = te.fit_transform(X_tr, y_tr)
        X_va_enc = te.transform(X_va)

        model = LogisticRegression(
            C=C,
            max_iter=1000,
            class_weight="balanced",
            solver="lbfgs"
        )
        model.fit(X_tr_enc, y_tr)
        oof_pred[va_idx] = model.predict_proba(X_va_enc)[:, 1]

    return roc_auc_score(y_array, oof_pred)


In [ ]:
1️⃣ Đoạn logic chọn cột & tạo transformer (dùng riêng cho Logistic)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# X_sel: DataFrame sau khi chọn feature (top_features)
# Bạn tự điền danh sách cột muốn one-hot ở đây
ONE_HOT_COLS = [
    "col_cat_1",
    "col_cat_2",
    "col_cat_3",
    # ...
]

# Tự động suy ra các cột còn lại (không one-hot)
num_cols = [c for c in X_sel.columns if c not in ONE_HOT_COLS]

# Bộ tiền xử lý:
# - với ONE_HOT_COLS  → OneHotEncoder
# - với num_cols      → giữ nguyên (passthrough)
preprocessor_for_logistic = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), ONE_HOT_COLS),
        ("num", "passthrough", num_cols),
    ]
)

# Pipeline Logistic:
logistic_pipeline = Pipeline([
    ("preprocess", preprocessor_for_logistic),
    ("scaler", StandardScaler(with_mean=False)),  # vì sau OHE ra sparse
    ("clf", LogisticRegression(
        C=1.0,
        penalty="l2",
        solver="lbfgs",
        max_iter=1000,
        class_weight="balanced",
        n_jobs=1
    ))
])


Đoạn này bạn đặt ở đầu file (sau khi có X_sel), chỉ logistic dùng, các model khác (LGB/XGB/RF) vẫn dùng X_sel bình thường, không one-hot.

2️⃣ Cách nhét vào objective_logreg (ví dụ với Optuna)

Nếu bạn muốn tune C bằng Optuna, chỉ cần thay phần logistic trong objective_logreg như sau:

def objective_logreg(trial):
    C = trial.suggest_float("C", 1e-3, 10.0, log=True)

    # clone pipeline để mỗi trial/mỗi fold là instance mới
    from sklearn.base import clone
    base_pipe = Pipeline([
        ("preprocess", preprocessor_for_logistic),
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(
            C=C,
            penalty="l2",
            solver="lbfgs",
            max_iter=1000,
            class_weight="balanced",
            n_jobs=1
        ))
    ])

    oof_pred = np.zeros(len(X_sel))

    for tr_idx, va_idx in folds.split(X_sel, y):
        X_tr, X_va = X_sel.iloc[tr_idx], X_sel.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        pipe = clone(base_pipe)
        pipe.fit(X_tr, y_tr)
        prob_va = pipe.predict_proba(X_va)[:, 1]
        oof_pred[va_idx] = prob_va

    auc = roc_auc_score(y, oof_pred)
    return auc

Tóm tắt:

Bạn tự liệt kê các cột cần one-hot vào ONE_HOT_COLS.

Các cột còn lại được hiểu là numeric / không one-hot.

Logic one-hot này chỉ dùng trong pipeline Logistic,
còn LightGBM / XGBoost / RandomForest thì không đụng gì tới OHE.

Nếu bạn gửi mình vài tên cột thật (ví dụ "gender", "city", "job_type"), mình có thể viết lại block ONE_HOT_COLS đúng tên cột dataset của bạn để bạn chỉ việc copy dán.

ChatGPT can make mistakes. Check important info. See Cookie Preferences.